# wilro-sync on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wiltechs-hd/wilro-sync/blob/main/notebooks/wilro_sync_colab.ipynb)

Train and run **[wilro-sync](https://github.com/wiltechs-hd/wilro-sync)** (OmniSync-style lip sync on Wan2.1, with
target-face control) on a Colab GPU. The code is cloned from GitHub, so you only need this notebook.

**Status:** no pretrained checkpoint has been released yet, so this notebook covers the whole loop:
set up → sanity checks → prepare your training clips → train → lip-sync a video with your checkpoint.

| Runtime | What works |
|---|---|
| **A100** (40/80 GB) | everything; fastest training |
| **L4** (24 GB) | training (LoRA, 512×512×49 frames, 8-bit Adam), data prep, inference |
| **T4** (16 GB) | tests, data prep, inference (fp16, experimental). **Not enough for training** (no bf16). |

Choose the runtime under **Runtime → Change runtime type**. High-RAM helps. The model downloads need about
30 GB of local disk (the umT5 text encoder alone is 22 GB) and go to `/content/hf_cache`, not your Drive.

Everything you want to keep (training clips, checkpoints, outputs) is written to a folder on your Google Drive,
so a disconnected session can resume where it stopped.

## 1. Settings

In [ ]:
# @title Paths and code source { display-mode: "form" }
CODE_SOURCE = "github"  # @param ["github", "drive", "zip", "local"]
# github: clone GITHUB_URL at GIT_REF (default; the repo is public)
# drive : copy your own working copy from a folder on Google Drive
# zip   : upload a .zip of the repo when prompted
# local : the repo is already at /content/wilro-sync
GITHUB_URL = "https://github.com/wiltechs-hd/wilro-sync.git"  # @param {type:"string"}
GIT_REF = "main"  # @param {type:"string"}
# GIT_REF: a branch, tag or commit to run (pin a commit to reproduce a training run)
DRIVE_CODE_DIR = "/content/drive/MyDrive/wilro-sync"  # @param {type:"string"}
WORK_DIR = "/content/drive/MyDrive/wilro-sync-work"  # @param {type:"string"}
# WORK_DIR keeps raw videos, pre-computed clips, checkpoints and outputs (persists on Drive)

import os
import subprocess
import sys

REPO = "/content/wilro-sync"
os.environ.setdefault("HF_HOME", "/content/hf_cache")  # big model files stay on local disk
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and (CODE_SOURCE == "drive" or WORK_DIR.startswith("/content/drive")):
    from google.colab import drive

    drive.mount("/content/drive")

for sub in ("raw/wild", "latents", "runs", "outputs", "manifests"):
    os.makedirs(os.path.join(WORK_DIR, sub), exist_ok=True)
RAW = os.path.join(WORK_DIR, "raw")
LATENTS = os.path.join(WORK_DIR, "latents")


def sh(cmd: str, check: bool = True) -> int:
    """Run a shell command and stream its output into the notebook."""
    print(f"$ {cmd}")
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()
    if check and rc != 0:
        raise RuntimeError(f"command failed with exit code {rc}")
    return rc


def latest_checkpoint(run_dir: str):
    import glob

    steps = sorted(glob.glob(os.path.join(run_dir, "step_*")))
    steps = [d for d in steps if os.path.isfile(os.path.join(d, "step.json"))]
    return steps[-1] if steps else None


print("work dir:", WORK_DIR)

In [ ]:
# @title Optional: Hugging Face token (faster, rate-limit-free downloads) { display-mode: "form" }
# Add a secret named HF_TOKEN in Colab (key icon on the left). Skipped if absent.
try:
    from google.colab import userdata

    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN set")
except Exception:
    print("no HF_TOKEN secret, continuing anonymously")

## 2. Check the GPU

In [ ]:
import torch

if torch.cuda.is_available():
    GPU = torch.cuda.get_device_name(0)
    GPU_MEM = torch.cuda.get_device_properties(0).total_memory / 2**30
    BF16 = torch.cuda.is_bf16_supported()
    sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv", check=False)
else:
    GPU, GPU_MEM, BF16 = "none", 0.0, False
print(f"GPU: {GPU}  memory: {GPU_MEM:.0f} GB  bf16: {BF16}")
CAN_TRAIN = BF16 and GPU_MEM >= 22
if not torch.cuda.is_available():
    print("No GPU: only the tests (section 4) will work. Switch the runtime to a GPU.")
elif not CAN_TRAIN:
    print("This GPU can run tests, data prep and inference, but not training. Use an L4 or A100 to train.")

## 3. Get the code and install

In [ ]:
import shutil

if CODE_SOURCE == "drive":
    if not os.path.isdir(DRIVE_CODE_DIR):
        raise FileNotFoundError(f"{DRIVE_CODE_DIR} not found: copy the wilro-sync folder to your Drive first")
    shutil.rmtree(REPO, ignore_errors=True)
    shutil.copytree(DRIVE_CODE_DIR, REPO, ignore=shutil.ignore_patterns(
        "__pycache__", "*.egg-info", ".git", "runs", "data", "Claude outputs"))
elif CODE_SOURCE == "github":
    if os.path.isdir(os.path.join(REPO, ".git")):  # re-run: update the existing clone
        sh(f"git -C {REPO} fetch --depth 1 origin {GIT_REF} && git -C {REPO} checkout -q FETCH_HEAD")
    else:
        shutil.rmtree(REPO, ignore_errors=True)
        sh(f"git clone -q {GITHUB_URL} {REPO} && git -C {REPO} checkout -q {GIT_REF}")
    sh(f"git -C {REPO} log -1 --format='code: %h %s (%cd)' --date=short")
elif CODE_SOURCE == "zip":
    from google.colab import files

    up = files.upload()
    zip_name = next(iter(up))
    shutil.rmtree(REPO, ignore_errors=True)
    sh(f"mkdir -p /content/_zip && unzip -q -o '{zip_name}' -d /content/_zip")
    root = next(os.path.dirname(p) for p, _, f in os.walk("/content/_zip") if "pyproject.toml" in f)
    shutil.move(root, REPO)
if not os.path.isfile(os.path.join(REPO, "pyproject.toml")):
    raise FileNotFoundError(f"no wilro-sync repo at {REPO}")

os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)  # editable installs are only picked up by new processes
extras = ".[dev,train]" if torch.cuda.is_available() else ".[dev]"
sh(f'pip install -q -e "{extras}"')
import wilrosync

print("wilro-sync", wilrosync.__version__, "installed from", REPO)

## 4. Sanity checks

* **Unit tests** (CPU, ~20 s): math, model wrapper, data prep, a tiny training run and a tiny multi-face inference.
* **Real-weights check** (~6 GB download, 1–2 min): confirms the lip-sync wrapper reproduces Wan2.1-1.3B
  exactly before training, and that the real VAE and Whisper produce the expected shapes.

In [ ]:
sh("python -m pytest -q")

In [ ]:
# @title Real Wan2.1-1.3B check { display-mode: "form" }
RUN_REAL_CHECK = True  # @param {type:"boolean"}
if RUN_REAL_CHECK:
    sh("python scripts/check_real_wan.py")

## 5. Prepare training clips

Put talking-head videos in **`WORK_DIR/raw/wild/`** (any `.mp4`/`.mov`/`.mkv`; one clearly visible speaker,
good audio-video sync, at least ~3 s each). More and more varied speakers give better results; start with a few
dozen to check that everything works, then scale up.

Optional **pseudo pairs** (paper §3.2, used for the high-noise steps): a file `WORK_DIR/raw/pairs.jsonl` with lines
like `{"kind": "pseudo_pair", "cond": "mead/M003/001.mp4", "target": "mead/M003/002.mp4"}` (same speaker and camera,
different sentence; paths relative to `WORK_DIR/raw`). Without pairs, training still runs with timestep-dependent
sampling switched off.

Each clip is cropped to the centre, so **frame the speaker's face near the centre** or crop your videos first
(face-centred crops come with a later milestone). Clips are written to `WORK_DIR/latents/` and reused by every
training run.

In [ ]:
# @title Clip settings { display-mode: "form" }
CLIP_SIZE = 512  # @param {type:"integer"}
CLIP_FRAMES = 49  # @param {type:"integer"}
CLIPS_PER_VIDEO = 4  # @param {type:"integer"}
CAPTION = "A person speaking clearly with natural, well-articulated lip and teeth movements."  # @param {type:"string"}
REBUILD = False  # @param {type:"boolean"}
assert (CLIP_FRAMES - 1) % 4 == 0, "CLIP_FRAMES must be 1 + 4k (e.g. 33, 49, 81)"

import glob
import json

VIDEO_EXT = (".mp4", ".mov", ".mkv", ".webm", ".avi")
videos = sorted(p for p in glob.glob(os.path.join(RAW, "wild", "**", "*"), recursive=True)
                if p.lower().endswith(VIDEO_EXT))
print(f"{len(videos)} videos in {RAW}/wild")
WILD_MANIFEST = os.path.join(WORK_DIR, "manifests", "wild.jsonl")
with open(WILD_MANIFEST, "w") as f:
    for p in videos:
        f.write(json.dumps({"kind": "arbitrary", "video": os.path.relpath(p, RAW), "caption": CAPTION}) + "\n")
PAIRS_MANIFEST = os.path.join(RAW, "pairs.jsonl")
HAS_PAIRS = os.path.isfile(PAIRS_MANIFEST)
print("pseudo-pair manifest:", PAIRS_MANIFEST if HAS_PAIRS else "none")

In [ ]:
# @title Run data preparation (GPU; first run downloads ~28 GB of encoders) { display-mode: "form" }
WILD_OUT, PAIRS_OUT = os.path.join(LATENTS, "wild"), os.path.join(LATENTS, "pairs")
dev = "cuda" if torch.cuda.is_available() else "cpu"


def prepare(manifest: str, out: str) -> None:
    index = os.path.join(out, "index.jsonl")
    if os.path.isfile(index) and not REBUILD:
        n = sum(1 for _ in open(index))
        print(f"{out}: {n} clips already prepared (tick REBUILD to redo)")
        return
    shutil.rmtree(out, ignore_errors=True)
    sh(f"python scripts/prepare_clips.py --manifest '{manifest}' --root '{RAW}' --out '{out}' "
       f"--size {CLIP_SIZE} {CLIP_SIZE} --frames {CLIP_FRAMES} --clips-per-video {CLIPS_PER_VIDEO} --device {dev}")


if not videos:
    raise RuntimeError(f"add some videos to {RAW}/wild first")
prepare(WILD_MANIFEST, WILD_OUT)
if HAS_PAIRS:
    prepare(PAIRS_MANIFEST, PAIRS_OUT)

## 6. Train

Defaults follow `configs/train/stage_a.yaml` (Wan2.1-1.3B + audio adapter + LoRA, lr 1e-5 / 1e-4 for new layers).
Checkpoints go to `WORK_DIR/runs/<RUN_NAME>/step_*`; with **RESUME** ticked, re-running this cell continues
from the latest one (useful after Colab disconnects).

A first run of a few hundred steps on a handful of clips is a good check that the loss falls. Real quality needs
many hours of varied data and tens of thousands of steps.

In [ ]:
# @title Training settings { display-mode: "form" }
RUN_NAME = "stage_a_colab"  # @param {type:"string"}
TRAIN_MODE = "lora"  # @param ["lora", "adapter", "full"]
MAX_STEPS = 2000  # @param {type:"integer"}
GRAD_ACCUM = 8  # @param {type:"integer"}
LORA_RANK = 64  # @param {type:"integer"}
SAVE_EVERY = 250  # @param {type:"integer"}
KEEP_LAST = 2  # @param {type:"integer"}
RESUME = True  # @param {type:"boolean"}

import json
import shlex

RUN_DIR = os.path.join(WORK_DIR, "runs", RUN_NAME)
PAIRS_INDEX = os.path.join(LATENTS, "pairs", "index.jsonl")
HAS_PAIRS = os.path.isfile(PAIRS_INDEX)
if not os.path.isfile(os.path.join(LATENTS, "wild", "index.jsonl")):
    raise FileNotFoundError("no prepared clips: run section 5 first")
overrides = {
    "output_dir": RUN_DIR,
    "data.arbitrary_index": os.path.join(LATENTS, "wild", "index.jsonl"),
    "data.pairs_index": PAIRS_INDEX if HAS_PAIRS else "null",
    "data.tds_enabled": str(HAS_PAIRS).lower(),
    "data.null_text": os.path.join(LATENTS, "null_text.safetensors"),
    "data.num_workers": 2,
    "train_mode": TRAIN_MODE,
    "lora_rank": LORA_RANK,
    "max_steps": MAX_STEPS,
    "grad_accum": GRAD_ACCUM,
    "save_every": SAVE_EVERY,
    "keep_last": KEEP_LAST,
    "mixed_precision": "bf16" if BF16 else "no",
    "optimizer": "adamw" if GPU_MEM >= 60 else "adamw8bit",
}
ckpt = latest_checkpoint(RUN_DIR) if RESUME else None
if ckpt:
    overrides["resume_from"] = ckpt
    print("resuming from", ckpt)
TRAIN_CMD = "wilro-sync train --config configs/train/stage_a.yaml " + " ".join(
    shlex.quote(f"{k}={v}") for k, v in overrides.items())
print(TRAIN_CMD)

In [ ]:
# @title Start / resume training { display-mode: "form" }
if not CAN_TRAIN:
    raise RuntimeError(f"{GPU} cannot train this model (needs bf16 and >= 22 GB). Switch to an L4 or A100 runtime.")
sh(TRAIN_CMD)

In [ ]:
# @title Loss curve { display-mode: "form" }
import json

import matplotlib.pyplot as plt

log_path = os.path.join(RUN_DIR, "log.jsonl")
if os.path.isfile(log_path):
    rows = [json.loads(l) for l in open(log_path) if l.strip()]
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.plot([r["step"] for r in rows], [r["loss"] for r in rows], lw=1.5)
    ax.set_xlabel("step")
    ax.set_ylabel("flow-matching loss")
    ax.set_title(RUN_NAME)
    ax.grid(alpha=0.3)
    plt.show()
    print(f"last: step {rows[-1]['step']}, loss {rows[-1]['loss']:.4f}, {rows[-1]['it_s']:.2f} it/s")
else:
    print("no log yet:", log_path)

## 7. Lip-sync a video

Pick a source video and a speech track (paths on your Drive, or upload them). For a scene with several faces,
set **BBOX** to the target face (`x0,y0,x1,y1` in pixels of the original video); the preview cell draws a grid
and your box to help. Leave it empty for single-speaker videos.

Until a checkpoint has been trained properly, the output will not be lip-synced; use this section to check the
pipeline and to follow training progress.

In [ ]:
# @title Inputs { display-mode: "form" }
RUN_NAME = "stage_a_colab"  # @param {type:"string"}
CKPT = "latest"  # @param {type:"string"}
USE_EMA = True  # @param {type:"boolean"}
VIDEO_PATH = ""  # @param {type:"string"}
AUDIO_PATH = ""  # @param {type:"string"}
UPLOAD = False  # @param {type:"boolean"}
BBOX = ""  # @param {type:"string"}

if UPLOAD:
    from google.colab import files

    print("upload the source video, then the audio")
    VIDEO_PATH = os.path.abspath(next(iter(files.upload())))
    AUDIO_PATH = os.path.abspath(next(iter(files.upload())))
RUN_DIR = os.path.join(WORK_DIR, "runs", RUN_NAME)
if CKPT == "latest":
    CKPT = latest_checkpoint(RUN_DIR)
    if CKPT is None:
        raise FileNotFoundError(f"no checkpoint in {RUN_DIR}; train first or set CKPT to a checkpoint folder")
if USE_EMA and os.path.isdir(os.path.join(CKPT, "ema")):
    CKPT = os.path.join(CKPT, "ema")
print("checkpoint:", CKPT)
for p in (VIDEO_PATH, AUDIO_PATH):
    if not os.path.isfile(p):
        raise FileNotFoundError(f"missing input: {p!r}")

In [ ]:
# @title Preview the first frame (with your target box) { display-mode: "form" }
import matplotlib.patches as patches
import matplotlib.pyplot as plt

from wilrosync.io.media import read_video

first, _fps = read_video(VIDEO_PATH, max_frames=1)
h, w = first.shape[1:3]
fig, ax = plt.subplots(figsize=(10, 10 * h / w))
ax.imshow(first[0])
ax.set_xticks(range(0, w, max(1, w // 10)))
ax.set_yticks(range(0, h, max(1, h // 10)))
ax.grid(alpha=0.4)
TARGET_BOX = None
if BBOX.strip():
    TARGET_BOX = tuple(float(v) for v in BBOX.split(","))
    x0, y0, x1, y1 = TARGET_BOX
    ax.add_patch(patches.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, lw=2, ec="lime"))
ax.set_title(f"{w}x{h}" + (f"  target box {TARGET_BOX}" if TARGET_BOX else "  (whole frame)"))
plt.show()

In [ ]:
# @title Run lip sync { display-mode: "form" }
STEPS = 50  # @param {type:"integer"}
TAU = 0.92  # @param {type:"number"}
OMEGA_PEAK = 4.5  # @param {type:"number"}
MAX_SIDE = 832  # @param {type:"integer"}
SEED = 0  # @param {type:"integer"}

import time

from wilrosync.io.media import read_audio, write_video
from wilrosync.pipeline.lipsync import LipSyncPipeline, load_infer_config
from wilrosync.pipeline.target import TargetSignals

dtype = torch.bfloat16 if BF16 else torch.float16
if globals().get("_PIPE_CKPT") != CKPT:  # load once, reuse across runs
    PIPE = LipSyncPipeline.from_pretrained(CKPT, device="cuda", dtype=dtype)
    _PIPE_CKPT = CKPT

frames, fps = read_video(VIDEO_PATH)
wav = read_audio(AUDIO_PATH)
target = TargetSignals.from_bbox(len(frames), *frames.shape[1:3], TARGET_BOX) if TARGET_BOX else None
cfg = load_infer_config("configs/infer/default.yaml")
cfg.steps, cfg.tau, cfg.max_side, cfg.seed = STEPS, TAU, MAX_SIDE, SEED
cfg.dscfg.omega_peak = OMEGA_PEAK

t0 = time.time()
out = PIPE(frames, fps, wav, target=target, cfg=cfg)
step_dir = os.path.dirname(CKPT) if os.path.basename(CKPT) == "ema" else CKPT
name = f"{os.path.splitext(os.path.basename(VIDEO_PATH))[0]}_{os.path.basename(step_dir)}.mp4"
OUT_PATH = os.path.join(WORK_DIR, "outputs", name)
write_video(OUT_PATH, out, fps, audio_path=AUDIO_PATH)
print(f"{len(out)} frames in {time.time() - t0:.0f} s -> {OUT_PATH}")

In [ ]:
# @title Watch the result { display-mode: "form" }
from base64 import b64encode

from IPython.display import HTML, display

size_mb = os.path.getsize(OUT_PATH) / 2**20
if size_mb < 60:
    data = b64encode(open(OUT_PATH, "rb").read()).decode()
    display(HTML(f'<video width="720" controls src="data:video/mp4;base64,{data}"></video>'))
else:
    print(f"{size_mb:.0f} MB is too large to embed; open it from Drive: {OUT_PATH}")

## Notes

* **Disconnected?** Re-run sections 1–3, then section 6 with RESUME ticked; clips and checkpoints are on Drive.
* **New code on GitHub?** Re-run section 3 to pull it (or set `GIT_REF` to a commit to stay on a fixed version).
* **Issues / questions:** https://github.com/wiltechs-hd/wilro-sync/issues
* **Out of memory while training:** lower `CLIP_SIZE` (e.g. 384) or `CLIP_FRAMES` (33) and re-prepare,
  or use `TRAIN_MODE="adapter"`.
* **Out of memory at inference:** lower `MAX_SIDE` (e.g. 640). The 11 GB text encoder is loaded only to encode
  the prompt and freed straight after.
* Only process footage you have the rights and consent to edit, and label the results as synthetic.